# 03 - Detector Evaluation, Checkpoint Reconciliation & Multi-Detector Separate Datasets (WORKING COPY)
**Master Workspace:** `MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/08_COLAB_NOTEBOOKS`  
**Original Notebook Preserved:** `03_Detector_Evaluation.ipynb`  

### Targeted Dataset Separation & Benchmark Fixes Applied in This Notebook:
1. **Preserved Historical `YOLO11` Dataset Untouched:**
   - `03_DATASET_AUDIT/YOLO11_DATASET_UNZIPPED/` (`data_colab.yaml`)
   - `03_DATASET_AUDIT/_BACKUP_YOLO11_DATASET_UNZIPPED_PREV/`
   - `03_DATASET_AUDIT/bima_lppm2_.v2i.yolov11.zip` (`1,373` train / `393` valid / `196` test = `1,962` images, `40,930` instances)
2. **Configured Separate `YOLOv8` Dataset (`1,373 / 393 / 196` Historical Parity):**
   - Dataset Root: `03_DATASET_AUDIT/DETECTOR_DATASETS/YOLOv8/`
   - Config YAML: `03_DATASET_AUDIT/DETECTOR_DATASETS/YOLOv8/data.yaml`
   - Isolated Output Folder: `09_REVALIDATED_EXPERIMENT_OUTPUTS/detector_benchmarks/yolov8m/`
3. **Configured Separate `RF-DETR` COCO JSON Dataset (`1,373 / 393 / 196` Historical Parity):**
   - Dataset Root: `03_DATASET_AUDIT/DETECTOR_DATASETS/RF_DETR/`
   - COCO JSONs: `train/_annotations.coco.json` (`1,373` images, `28,653` boxes), `valid/_annotations.coco.json` (`393` images, `8,189` boxes), `test/_annotations.coco.json` (`196` images, `4,088` boxes)
   - Isolated Output Folder: `09_REVALIDATED_EXPERIMENT_OUTPUTS/detector_benchmarks/rfdetr_small/`
4. **Execution Locks Active by Default:** `ENABLE_MODEL_EVALUATION = False`, `ENABLE_TRAINING = False`, and `ENABLE_RFDETR_TRAINING = False` prevent accidental GPU training or evaluation until explicitly enabled.

In [1]:
# ==============================================================================
# CENTRALIZED GOOGLE DRIVE PATHS, EXECUTION LOCKS & MULTI-DETECTOR DATASETS
# ==============================================================================
import json
import logging
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Execution Locks (Set to True in Colab when ready to execute)
ENABLE_ROBOFLOW_STAGING_DOWNLOAD = False
ENABLE_ROBOFLOW_COCO_DOWNLOAD = False
ENABLE_DATASET_REPLACEMENT = False
ENABLE_MODEL_EVALUATION = False
ENABLE_TRAINING = False
ENABLE_RFDETR_TRAINING = True

# Selector for RF-DETR Model Size:
#   "S"    -> RF-DETR-Small (rfdetr.RFDETRSmall, saved to detector_benchmarks/rfdetr_small/)
#   "L"    -> RF-DETR-Large (rfdetr.RFDETRLarge, saved to detector_benchmarks/rfdetr_large/)
#   "BOTH" -> Runs both RF-DETR-Small and RF-DETR-Large sequentially into their own folders
RFDETR_MODEL_SIZE = "L"
# Speed & Convergence Settings for RF-DETR and YOLOv8:
#   USE_LOCAL_COLAB_SSD_CACHE = True copies dataset from slow Google Drive FUSE to /content/local_detector_datasets (5x-10x faster I/O)
#   RFDETR_EPOCHS = 35 (with early_stopping=True, patience=8) converges DINOv2-pretrained RF-DETR in ~25-45 mins instead of >10 hours
USE_LOCAL_COLAB_SSD_CACHE = True
RFDETR_EPOCHS = 100
RFDETR_EARLY_STOPPING = True
RFDETR_PATIENCE = 8
YOLOV8_EPOCHS = 150

ENABLE_FULL_VIDEO_INFERENCE = False
ENABLE_TRACKER_REPLAY = False
ENABLE_MCDM_EVALUATION = False
ENABLE_IDENTITY_ABLATION = False
ENABLE_SEAT_PERTURBATION_SIMULATION = False
ENABLE_TEMPORAL_EXECUTION = False
ENABLE_MONTE_CARLO = False
ENABLE_GPU_EXPERIMENTS = False

# Default Roboflow API Key for SDN Ngaliyan Version 1 (COCO / YOLOv11)
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "RMForwlhIZfWVNZ1AGnc")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s")
logger = logging.getLogger("YOLO11_IJITCS_SyncedColab")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MASTER_ROOT = Path(
    "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026"
)

# 1. YOLO11 Preserved Dataset Paths (Untouched)
DATASET_AUDIT_DIR = MASTER_ROOT / "03_DATASET_AUDIT"
DATASET_ROOT = DATASET_AUDIT_DIR / "YOLO11_DATASET_UNZIPPED"
DATA_YAML_PATH = DATASET_ROOT / "data_colab.yaml"
STAGING_DIR = DATASET_AUDIT_DIR / "_STAGING_ROBOFLOW_YOLOV11"
BACKUP_DATASET_DIR = DATASET_AUDIT_DIR / "_BACKUP_YOLO11_DATASET_UNZIPPED_PREV"
ZIP_ARCHIVE_PATH = DATASET_AUDIT_DIR / "bima_lppm2_.v2i.yolov11.zip"
DATASET_ZIP_PATH = ZIP_ARCHIVE_PATH
DATASET_PATH = DATASET_ROOT
DATA_ROBOFLOW_ORIG_YAML_PATH = DATASET_ROOT / "data_roboflow_original.yaml"

# 2. Separate Detector Datasets (Historical 1,373 / 393 / 196 Split Parity)
DETECTOR_DATASETS_DIR = DATASET_AUDIT_DIR / "DETECTOR_DATASETS"
YOLOV8_DATASET_DIR = DETECTOR_DATASETS_DIR / "YOLOv8"
YOLOV8_DATA_YAML_PATH = YOLOV8_DATASET_DIR / "data.yaml"
RFDETR_DATASET_DIR = DETECTOR_DATASETS_DIR / "RF_DETR"
RFDETR_MANIFEST_PATH = RFDETR_DATASET_DIR / "rf_detr_dataset_manifest.json"
PARITY_REPORT_CSV_PATH = DETECTOR_DATASETS_DIR / "DETECTOR_DATASET_SPLIT_PARITY_REPORT.csv"

# 3. Code, Checkpoints & Strictly Separated Per-Detector Output Directories
REPO_ROOT = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "YOLO11_Repository"
MODULES_DIR = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "Reconstructed_Research_Modules"
VIDEO_PATH = MASTER_ROOT / "02_ORIGINAL_RESEARCH_ARCHIVE" / "Original_Videos" / "video_part001.mp4"
CHECKPOINT_DIR = MASTER_ROOT / "04_MODEL_AND_CONFIGURATION" / "Checkpoints"

OUTPUT_DIR = MASTER_ROOT / "09_REVALIDATED_EXPERIMENT_OUTPUTS"
OUTPUT_PATH = OUTPUT_DIR
BENCHMARK_OUTPUT_ROOT = OUTPUT_DIR / "detector_benchmarks"
YOLO11_BENCHMARK_OUT_DIR = BENCHMARK_OUTPUT_ROOT / "yolo11m"
YOLOV8_BENCHMARK_OUT_DIR = BENCHMARK_OUTPUT_ROOT / "yolov8m"
RFDETR_SMALL_BENCHMARK_OUT_DIR = BENCHMARK_OUTPUT_ROOT / "rfdetr_small"
RFDETR_LARGE_BENCHMARK_OUT_DIR = BENCHMARK_OUTPUT_ROOT / "rfdetr_large"
RTDETR_BENCHMARK_OUT_DIR = BENCHMARK_OUTPUT_ROOT / "rtdetr_l"
MULTI_DETECTOR_PIPELINE_DIR = OUTPUT_DIR / "multi_detector_pipeline"

PRIMARY_BASELINE_CHECKPOINT = CHECKPOINT_DIR / "best.pt"
BASELINE_BEST_CHECKPOINT = CHECKPOINT_DIR / "yolo11m_baseline_best.pt"
FOCAL_BEST_CHECKPOINT = CHECKPOINT_DIR / "yolo11m_focal_best.pt"
BALANCED_BEST_CHECKPOINT = CHECKPOINT_DIR / "yolo11m_balanced_best.pt"
PRETRAINED_YOLO11M_PATH = CHECKPOINT_DIR / "yolo11m.pt"
PRETRAINED_YOLOV8M_PATH = CHECKPOINT_DIR / "yolov8m.pt"
PRETRAINED_RFDETR_SMALL_PATH = CHECKPOINT_DIR / "rf-detr-small.pth"
PRETRAINED_RFDETR_LARGE_PATH = CHECKPOINT_DIR / "rf-detr-large.pth"

if IN_COLAB and MASTER_ROOT.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for _d in (
        YOLO11_BENCHMARK_OUT_DIR,
        YOLOV8_BENCHMARK_OUT_DIR,
        RFDETR_SMALL_BENCHMARK_OUT_DIR,
        RFDETR_LARGE_BENCHMARK_OUT_DIR,
        RTDETR_BENCHMARK_OUT_DIR,
        MULTI_DETECTOR_PIPELINE_DIR / "yolov8m",
        MULTI_DETECTOR_PIPELINE_DIR / "rfdetr_small",
        MULTI_DETECTOR_PIPELINE_DIR / "rfdetr_large",
    ):
        _d.mkdir(parents=True, exist_ok=True)

# Register REPO_ROOT and MODULES_DIR on sys.path
for _p in (REPO_ROOT, MODULES_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
importlib.invalidate_caches()
for _m in list(sys.modules.keys()):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

# Ensure roboflow, ultralytics, and rfdetr[train,loggers] + pytorch_lightning are installed in Colab
if IN_COLAB:
    for _pkg_import, _pip_spec in (
        ("roboflow", "roboflow"),
        ("ultralytics", "ultralytics"),
        ("pytorch_lightning", "pytorch_lightning"),
        ("pycocotools", "pycocotools"),
        ("supervision", "supervision"),
        ("rfdetr", "rfdetr[train,loggers]"),
    ):
        try:
            __import__(_pkg_import)
        except ModuleNotFoundError:
            logger.info("Installing missing Colab dependency: %s ...", _pip_spec)
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pip_spec])

from src.dataset_manager import (
    HISTORICAL_SPLIT_COUNTS,
    download_roboflow_coco_to_staging,
    ensure_canonical_dataset_ready,
    ensure_rfdetr_dataset_ready,
    ensure_yolov8_dataset_ready,
    inspect_ultralytics_labels_cache,
    validate_coco_json_split,
    validate_yolo_dataset_structure,
)
from src.detector_adapters_v2_aligned import (
    EXPECTED_CLASS_MAPPING,
    HISTORICAL_CHECKPOINT_RECONCILIATION,
    ensure_rfdetr_train_dependencies,
    verify_environment_and_inputs,
    verify_yolo11_checkpoints,
)

logger.info("YOLO11 Dataset (Preserved): %s | YAML: %s", DATASET_ROOT, DATA_YAML_PATH)
logger.info("YOLOv8 Dataset (Separate):  %s | YAML: %s", YOLOV8_DATASET_DIR, YOLOV8_DATA_YAML_PATH)
logger.info("RF-DETR Dataset (Separate): %s | Active RFDETR_MODEL_SIZE: %s", RFDETR_DATASET_DIR, RFDETR_MODEL_SIZE)

Mounted at /content/drive


## Step 2: Reconcile Historical YOLO11m Checkpoints (`baseline` vs `focal` vs `balanced`)


In [2]:
# ==============================================================================
# HISTORICAL CHECKPOINT RECONCILIATION (`baseline` vs `focal` vs `balanced`)
# ==============================================================================
ckpt_summary_df = pd.DataFrame(HISTORICAL_CHECKPOINT_RECONCILIATION)
if ENABLE_MODEL_EVALUATION:
    ckpt_verification = verify_yolo11_checkpoints(CHECKPOINT_DIR)
    logger.info("Checkpoint Verification: %s", json.dumps(ckpt_verification, indent=2))
    ckpt_summary_df.to_csv(OUTPUT_DIR / "yolo11_checkpoint_reconciliation_verified.csv", index=False)
else:
    logger.info("[LOCKED] Checkpoint verification & CSV export ready (set ENABLE_MODEL_EVALUATION = True in Cell 1 to execute).")


## Step 3: Multi-Detector Separate Dataset Verification & Guarded Benchmark Suite (`YOLO11m`, `YOLOv8m`, `RF-DETR-Small`, `RT-DETR-L`)

In [3]:
# ==============================================================================
# MULTI-DETECTOR SEPARATE DATASET VERIFICATION & BENCHMARK SUITE
# ==============================================================================
from src.detector_adapters_v2_aligned import (
    RFDETRAdapter,
    RFDETRLargeAdapter,
    RFDETRSmallAdapter,
    RTDETRLAdapter,
    UnifiedDetectorBenchmarkSuite,
    YOLO11Adapter,
    YOLOv8mAdapter,
    ensure_rfdetr_train_dependencies,
    evaluate_all_yolo11_checkpoints,
    evaluate_checkpoint_on_split,
)

env_check = verify_environment_and_inputs(MASTER_ROOT)
print("=== Multi-Detector Environment & Separate Dataset Verification ===")
print(json.dumps(env_check, indent=2))

if PARITY_REPORT_CSV_PATH.exists():
    df_parity = pd.read_csv(PARITY_REPORT_CSV_PATH)
    print("\n=== Detector Dataset Split Parity Report (1,373 / 393 / 196) ===")
    print(df_parity[["Detector_Dataset", "Split", "Image_Count", "Total_Instances", "Class_0_Angkat_Tangan", "Class_1_Berdiri", "Class_2_Diam", "Class_3_Tidur", "Split_Filename_Membership_Parity"]].to_string(index=False))

suite = UnifiedDetectorBenchmarkSuite(MASTER_ROOT)
if ENABLE_MODEL_EVALUATION or ENABLE_TRAINING or ENABLE_RFDETR_TRAINING:
    if ENABLE_TRAINING:
        v8_ready = ensure_yolov8_dataset_ready(
            yolov8_dir=YOLOV8_DATASET_DIR,
            historical_zip_path=ZIP_ARCHIVE_PATH,
            historical_backup_dir=BACKUP_DATASET_DIR,
        )
        logger.info("YOLOv8 Separate Dataset Readiness: %s", v8_ready["status"])
    if ENABLE_RFDETR_TRAINING:
        ensure_rfdetr_train_dependencies()
        rf_ready = ensure_rfdetr_dataset_ready(
            rfdetr_dir=RFDETR_DATASET_DIR,
            yolov8_dir=YOLOV8_DATASET_DIR,
            historical_zip_path=ZIP_ARCHIVE_PATH,
            prefer_roboflow_coco_download=ENABLE_ROBOFLOW_COCO_DOWNLOAD,
            roboflow_api_key=ROBOFLOW_API_KEY,
        )
        logger.info("RF-DETR Separate Dataset Readiness: %s", rf_ready["status"])

    benchmark_report = suite.run_guarded_benchmark(
        data_yaml=DATA_YAML_PATH,
        yolov8_data_yaml=YOLOV8_DATA_YAML_PATH,
        rfdetr_dataset_dir=RFDETR_DATASET_DIR,
        rfdetr_model_size=RFDETR_MODEL_SIZE,
        yolov8_epochs=YOLOV8_EPOCHS,
        rfdetr_epochs=RFDETR_EPOCHS,
        rfdetr_early_stopping=RFDETR_EARLY_STOPPING,
        rfdetr_patience=RFDETR_PATIENCE,
        use_local_ssd_cache=USE_LOCAL_COLAB_SSD_CACHE,
        seed=42,
        enable_eval=ENABLE_MODEL_EVALUATION,
        enable_training=ENABLE_TRAINING,
        enable_rfdetr_training=ENABLE_RFDETR_TRAINING,
    )
    print(json.dumps(benchmark_report, indent=2))
else:
    readiness_manifest = suite.run_guarded_benchmark(
        data_yaml=DATA_YAML_PATH,
        yolov8_data_yaml=YOLOV8_DATA_YAML_PATH,
        rfdetr_dataset_dir=RFDETR_DATASET_DIR,
        rfdetr_model_size=RFDETR_MODEL_SIZE,
        yolov8_epochs=YOLOV8_EPOCHS,
        rfdetr_epochs=RFDETR_EPOCHS,
        rfdetr_early_stopping=RFDETR_EARLY_STOPPING,
        rfdetr_patience=RFDETR_PATIENCE,
        use_local_ssd_cache=USE_LOCAL_COLAB_SSD_CACHE,
        seed=42,
        enable_eval=False,
        enable_training=False,
        enable_rfdetr_training=False,
    )
    print("\n=== Guarded Multi-Detector Readiness Manifest (Execution Locked) ===")
    print(json.dumps(readiness_manifest, indent=2))

=== Multi-Detector Environment & Separate Dataset Verification ===
{
  "master_root_exists": true,
  "dataset_zip_exists": true,
  "yolo11_dataset_preserved_exists": true,
  "yolo11_data_colab_yaml_exists": true,
  "resolved_yolo11_data_yaml": "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/03_DATASET_AUDIT/YOLO11_DATASET_UNZIPPED/data_colab.yaml",
  "yolov8_separate_dataset_exists": true,
  "yolov8_data_yaml_exists": true,
  "resolved_yolov8_data_yaml": "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/03_DATASET_AUDIT/DETECTOR_DATASETS/YOLOv8/data.yaml",
  "rfdetr_separate_dataset_exists": true,
  "rfdetr_coco_train_exists": true,
  "rfdetr_coco_valid_exists": true,
  "rfdetr_coco_test_exists": true,
  "resolved_rfdetr_dataset_dir": "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/03_DATASET_AUDIT/DETECTOR_DATASETS/RF_DETR",
  "checkpoints_dir_exists": true,
  "class_mapping_valid": true,
  "isolated_output_dirs": {
    "yolo11m": "/conten

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-16 05:12:40] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-large-2026.pth


/root/.roboflow/models/rf-detr-large-2026.pth:   0%|          | 0.00/130M [00:00<?, ?iB/s]

[2026-11-16 05:12:42] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-large-2026.pth
[2026-11-16 05:12:43] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.
[2026-11-16 05:12:45] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-11-16 05:12:45] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 4. The detection head will be re-initialized to 4 classes.
INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


... (48 outputs left out)

[2026-11-16 05:56:51] [WARNING] rf-detr - Checkpoint has 4 classes but model is configured for 90. Using checkpoint class count (4). Pass num_classes=4 to suppress this warning.
[2026-11-16 05:56:51] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.60s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.79s).
Accumulating evaluation results...
DONE (t=0.73s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.794
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.977
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.930
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.803
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.793
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.457
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.698
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

[2026-11-16 05:57:17] [WARNING] rf-detr - Checkpoint has 4 classes but model is configured for 90. Using checkpoint class count (4). Pass num_classes=4 to suppress this warning.
[2026-11-16 05:57:17] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.52s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.66s).
Accumulating evaluation results...
DONE (t=0.38s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.813
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.979
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.945
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.814
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.816
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.493
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.736
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet